<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_SINGULARITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Jul 27 11:36:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ============================================================================
# TOPO-2026 6×5 COMPLETE METRICS WITH SINGULARITY - FULL EXPERIMENT
# ============================================================================
# This runs the actual GPT-OSS-20B experiment and computes 6×5 metrics
# ============================================================================

import os
import gc
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

@dataclass
class SingularityConfig:
    """Configuration for singularity computation with real LLM."""
    base_model_id: str = "openai/gpt-oss-20b"
    hidden_size: int = 2880
    dtype: torch.dtype = torch.bfloat16
    tasks: List[str] = field(default_factory=lambda: ['A', 'B', 'C'])
    task_names: Dict[str, str] = field(default_factory=lambda: {
        'A': 'World vs Sports',
        'B': 'Business vs Sci/Tech',
        'C': 'World vs Sci/Tech'
    })
    epochs_per_task: int = 6
    batch_size: int = 16
    learning_rates: List[Tuple[float, float]] = field(default_factory=lambda: [
        (5e-3, 1e-3),   # Run 0
        (1e-3, 5e-4),   # Run 1
        (1e-2, 2e-3),   # Run 2
        (5e-3, 5e-3),   # Run 3
        (2e-3, 1e-3)    # Run 4
    ])
    num_runs: int = 5
    seed: int = 123
    prime_limit: int = 13
    anchor_primes: List[int] = field(default_factory=lambda: [2, 3, 5, 7, 11, 13])
    memory_fidelity_threshold: float = 0.90
    validation_threshold: float = 1.0
    forward_transfer_threshold: float = 1.0
    autonomy_ratio_threshold: float = 1.0
    singularity_threshold: float = 0.5


# ============================================================================
# 2. DATASET UTILITIES
# ============================================================================

class AGNewsStreamDataset(Dataset):
    """AG News dataset for continual learning tasks."""

    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


def prepare_ag_news_tasks(tokenizer, config: SingularityConfig) -> Dict:
    """Prepare AG News tasks for continual learning."""
    print("[Dataset] Loading AG News...")
    raw_dataset = load_dataset('SetFit/ag_news', split='train')
    raw_test = load_dataset('SetFit/ag_news', split='test')

    def isolate_task(dataset, class_labels, sample_limit):
        filtered = dataset.filter(lambda x: x['label'] in class_labels)
        sampled = filtered.select(range(min(sample_limit, len(filtered))))
        texts = [item['text'] for item in sampled]
        labels = [item['label'] % 2 for item in sampled]
        return texts, labels

    task_defs = {
        'A': ([0, 1], 500),
        'B': ([2, 3], 1000),
        'C': ([0, 3], 1000)
    }

    datasets = {}
    for task, (class_labels, limit) in task_defs.items():
        texts, labels = isolate_task(raw_dataset, class_labels, limit)
        test_texts, test_labels = isolate_task(raw_test, class_labels, 200)

        tokens = tokenizer(texts, max_length=64, padding='max_length',
                          truncation=True, return_tensors='pt')
        test_tokens = tokenizer(test_texts, max_length=64, padding='max_length',
                               truncation=True, return_tensors='pt')

        datasets[task] = {
            'train': AGNewsStreamDataset(
                tokens.input_ids, tokens.attention_mask,
                torch.tensor(labels, dtype=torch.long)
            ),
            'test': AGNewsStreamDataset(
                test_tokens.input_ids, test_tokens.attention_mask,
                torch.tensor(test_labels, dtype=torch.long)
            ),
            'texts': texts,
            'test_texts': test_texts
        }

    print(f"[Dataset] Created {len(datasets)} tasks")
    for task, data in datasets.items():
        print(f"  Task {task}: {len(data['train'])} train, {len(data['test'])} test")

    return datasets


# ============================================================================
# 3. LLM ARCHITECTURE WITH TASK HEADS
# ============================================================================

class GPT_OSS_20B_TaskAware(nn.Module):
    """GPT-OSS-20B with task-specific classifier heads."""

    def __init__(self, base_model: nn.Module, hidden_size: int = 2880):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size

        device = next(base_model.parameters()).device
        dtype = next(base_model.parameters()).dtype

        self.classifier_A = nn.Linear(hidden_size, 2, dtype=dtype).to(device)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=dtype).to(device)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=dtype).to(device)

        self.current_task = 'A'
        self._init_heads()

    def _init_heads(self):
        for head in [self.classifier_A, self.classifier_B, self.classifier_C]:
            nn.init.xavier_uniform_(head.weight)
            nn.init.zeros_(head.bias)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            pooled = hidden_states[batch_idx, seq_lens, :]
        else:
            pooled = hidden_states[:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

    def freeze_previous_heads(self, task: str):
        if task == 'B':
            self.classifier_A.requires_grad_(False)
        elif task == 'C':
            self.classifier_B.requires_grad_(False)

    def reset_heads(self):
        device = next(self.base_model.parameters()).device
        dtype = next(self.base_model.parameters()).dtype

        self.classifier_A = nn.Linear(self.hidden_size, 2, dtype=dtype).to(device)
        self.classifier_B = nn.Linear(self.hidden_size, 2, dtype=dtype).to(device)
        self.classifier_C = nn.Linear(self.hidden_size, 2, dtype=dtype).to(device)
        self._init_heads()
        self.current_task = 'A'


# ============================================================================
# 4. TOPOLOGICAL GOVERNOR
# ============================================================================

class TopologicalGovernor:
    """TOPO-2026 Topological Governor for memory permanence."""

    def __init__(self, embed_layer: nn.Embedding, primes: List[int]):
        self.embed_layer = embed_layer
        self.anchor_indices = [p for p in primes if p < embed_layer.weight.shape[0]]
        self.snapshot = {}

        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])
        print(f"[Governor] Anchors: {self.anchor_indices}, Safety: {self.safety_constant:.4f}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# ============================================================================
# 5. LLM TRAINER
# ============================================================================

class LLMSingularityTrainer:
    """Trains GPT-OSS-20B and computes singularity metrics."""

    def __init__(self, config: SingularityConfig):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"[Trainer] Using device: {self.device}")

        self._load_model()
        self.tokenizer = AutoTokenizer.from_pretrained(
            config.base_model_id, trust_remote_code=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.datasets = prepare_ag_news_tasks(self.tokenizer, config)
        self.results = []
        self.start_time = time.time()

    def _load_model(self):
        print("[Trainer] Loading GPT-OSS-20B...")
        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_id,
            trust_remote_code=True,
            torch_dtype=self.config.dtype
        ).to(self.device)

        for param in self.base_model.parameters():
            param.requires_grad = False

        # Find embedding layer
        if hasattr(self.base_model, 'transformer') and hasattr(self.base_model.transformer, 'wte'):
            self.embed_layer = self.base_model.transformer.wte
        else:
            for module in self.base_model.modules():
                if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100000:
                    self.embed_layer = module
                    break

        self.embed_layer.weight.requires_grad = True
        self.model = GPT_OSS_20B_TaskAware(self.base_model, self.config.hidden_size)
        self.original_embed_weights = self.embed_layer.weight.detach().clone()

    def train_task(self, task: str, lr_embed: float, lr_cls: float,
                   governor: Optional[TopologicalGovernor] = None,
                   run_id: int = 0) -> Dict:
        """Train a single task."""
        dataset = self.datasets[task]['train']
        dataloader = DataLoader(dataset, batch_size=self.config.batch_size, shuffle=True)

        self.model.switch_task(task)
        self.model.train()

        head = getattr(self.model, f'classifier_{task}')
        optimizer = torch.optim.AdamW([
            {'params': self.embed_layer.weight, 'lr': lr_embed},
            {'params': head.parameters(), 'lr': lr_cls}
        ])

        total_steps = self.config.epochs_per_task * len(dataloader)
        desc = f"[Run {run_id}] Task {task}: lr_embed={lr_embed:.0e}, lr_cls={lr_cls:.0e}"

        losses = []
        with tqdm(total=total_steps, desc=desc, leave=True) as pbar:
            for epoch in range(self.config.epochs_per_task):
                epoch_losses = []
                for batch in dataloader:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)

                    optimizer.zero_grad()
                    logits = self.model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = F.cross_entropy(logits, labels)
                    loss.backward()

                    if governor:
                        governor.zero_anchor_gradients()

                    torch.nn.utils.clip_grad_norm_(self.embed_layer.weight, max_norm=1.0)
                    optimizer.step()

                    if governor:
                        governor.enforce_anchors()

                    epoch_losses.append(loss.item())
                    pbar.update(1)
                    pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

                losses.append(np.mean(epoch_losses))

        test_acc = self.evaluate_task(task)

        return {
            'task': task,
            'lr_embed': lr_embed,
            'lr_cls': lr_cls,
            'losses': losses,
            'final_loss': losses[-1] if losses else float('inf'),
            'test_accuracy': test_acc
        }

    def evaluate_task(self, task: str) -> float:
        """Evaluate model on a specific task."""
        self.model.switch_task(task)
        self.model.eval()

        dataset = self.datasets[task]['test']
        dataloader = DataLoader(dataset, batch_size=self.config.batch_size, shuffle=False)

        correct = 0
        total = 0

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                logits = self.model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(logits, dim=-1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return float(correct / total)

    def run_continual_learning_experiment(self, run_id: int, lr_embed: float, lr_cls: float) -> Dict:
        """Run full continual learning experiment."""
        print(f"\n{'='*60}")
        print(f"RUN {run_id + 1}/{self.config.num_runs}")
        print(f"Learning Rates: embed={lr_embed:.0e}, cls={lr_cls:.0e}")
        print('='*60)

        run_start = time.time()

        # Reset state
        torch.manual_seed(self.config.seed + run_id)
        self.model.reset_heads()
        with torch.no_grad():
            self.embed_layer.weight.copy_(self.original_embed_weights)

        metrics = {
            'run_id': run_id,
            'lr_embed': lr_embed,
            'lr_cls': lr_cls,
            'accuracies': {},
            'losses': {}
        }

        # Zero-shot evaluation
        print("\n[Zero-shot] Evaluating tasks before training...")
        zero_shot = {}
        for task in self.config.tasks:
            acc = self.evaluate_task(task)
            zero_shot[task] = acc
            print(f"  {task}: {acc:.2%}")
        metrics['zero_shot'] = zero_shot

        # Task A
        print(f"\n[Task A] Training...")
        result_a = self.train_task('A', lr_embed, lr_cls, governor=None, run_id=run_id)
        metrics['after_A'] = self.evaluate_task('A')
        print(f"  Accuracy after A: {metrics['after_A']:.2%}")

        # Create governor and snapshot
        governor = TopologicalGovernor(self.embed_layer, self.config.anchor_primes)
        governor.take_snapshot()
        self.model.freeze_previous_heads('B')

        # Task B
        print(f"\n[Task B] Training...")
        result_b = self.train_task('B', lr_embed, lr_cls, governor=governor, run_id=run_id)
        metrics['after_B'] = {
            'A': self.evaluate_task('A'),
            'B': self.evaluate_task('B')
        }
        print(f"  Accuracy A after B: {metrics['after_B']['A']:.2%}")
        print(f"  Accuracy B after B: {metrics['after_B']['B']:.2%}")

        self.model.freeze_previous_heads('C')

        # Task C
        print(f"\n[Task C] Training...")
        result_c = self.train_task('C', lr_embed, lr_cls, governor=governor, run_id=run_id)
        metrics['final'] = {
            'A': self.evaluate_task('A'),
            'B': self.evaluate_task('B'),
            'C': self.evaluate_task('C')
        }
        print(f"  Final A: {metrics['final']['A']:.2%}")
        print(f"  Final B: {metrics['final']['B']:.2%}")
        print(f"  Final C: {metrics['final']['C']:.2%}")

        # Verify governor integrity
        assert governor.verify_integrity(), "Governor integrity violated!"

        # Compute derived metrics
        forgetting_A = metrics['after_A'] - metrics['final']['A']
        forgetting_B = metrics['after_B']['B'] - metrics['final']['B']
        forgetting_avg = (forgetting_A + forgetting_B) / 2

        metrics['forgetting_avg'] = forgetting_avg
        metrics['memory_fidelity'] = 1.0 - forgetting_avg

        fwt_A = metrics['after_A'] - metrics['zero_shot']['A']
        fwt_B = metrics['after_B']['B'] - metrics['zero_shot']['B']
        fwt_C = metrics['final']['C'] - metrics['zero_shot']['C']
        fwt_avg = (fwt_A + fwt_B + fwt_C) / 3

        metrics['fwt_avg'] = fwt_avg
        metrics['forward_transfer'] = 1.0 + fwt_avg

        metrics['consistency'] = (metrics['final']['A'] + metrics['final']['B'] + metrics['final']['C']) / 3

        run_time = time.time() - run_start
        metrics['run_time_seconds'] = run_time

        print(f"\n{'─'*50}")
        print(f"RUN {run_id} COMPLETE ({run_time:.1f}s)")
        print(f"  Memory Fidelity: {metrics['memory_fidelity']:.4f}")
        print(f"  Forward Transfer: {metrics['forward_transfer']:.4f}")
        print(f"  Consistency: {metrics['consistency']:.4f}")
        print(f"{'─'*50}")

        return metrics

    def run_full_experiment(self) -> List[Dict]:
        """Run full multi-run experiment."""
        self.results = []

        for run_id, (lr_embed, lr_cls) in enumerate(self.config.learning_rates):
            result = self.run_continual_learning_experiment(run_id, lr_embed, lr_cls)
            self.results.append(result)

            # Cleanup
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        total_time = time.time() - self.start_time
        print(f"\n{'='*60}")
        print(f"EXPERIMENT COMPLETE - Total time: {total_time/60:.1f} minutes")
        print('='*60)

        return self.results


# ============================================================================
# 6. TOPO2026_6x5_METRICS CLASS
# ============================================================================

@dataclass
class TOPO2026_6x5_Metrics:
    """Complete 6 metrics × 5 runs certification system with singularity."""

    forgetting_rate: float = 0.0
    forgetting_std: float = 0.0
    forgetting_pass: bool = False
    forgetting_threshold: float = 10.0

    bwt_corrected: float = 0.0
    bwt_std: float = 0.0
    bwt_pass: bool = False
    bwt_threshold: float = -5.0

    fwt: float = 0.0
    fwt_std: float = 0.0
    fwt_pass: bool = False
    fwt_threshold: float = 20.0

    degradation: float = 0.0
    degradation_std: float = 0.0
    degradation_pass: bool = False
    degradation_threshold: float = 5.0

    consistency: float = 0.0
    consistency_std: float = 0.0
    consistency_pass: bool = False
    consistency_threshold: float = 85.0

    five_by_five_certified: bool = False

    singularity_value: float = 0.0
    singularity_std: float = 0.0
    singularity_status: str = "Not Started"
    singularity_threshold: float = 0.5

    agi_gate: float = 0.0
    intelligence_rate: float = 0.0
    memory_fidelity: float = 0.0
    memory_fidelity_std: float = 0.0
    validation_accuracy: float = 0.0
    forward_transfer: float = 0.0
    forward_transfer_std: float = 0.0
    compute_efficiency: float = 0.0
    autonomy: float = 0.0

    raw_metrics: Dict = field(default_factory=dict)
    run_data: List[Dict] = field(default_factory=list)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def compute_singularity(self):
        """Compute singularity value from components."""
        self.autonomy = 1.0 if self.intelligence_rate > 1.0 else 0.0

        self.singularity_value = (
            self.agi_gate *
            self.intelligence_rate *
            self.memory_fidelity *
            self.validation_accuracy *
            self.forward_transfer *
            self.compute_efficiency *
            self.autonomy
        )

        if self.singularity_value > 0:
            relative_std = np.sqrt(
                (self.memory_fidelity_std / self.memory_fidelity) ** 2 +
                (self.forward_transfer_std / self.forward_transfer) ** 2
            )
            self.singularity_std = self.singularity_value * relative_std
        else:
            self.singularity_std = 0.0

        if self.agi_gate < 0.5:
            self.singularity_status = "❌ No AGI (Pre-singularity)"
        elif self.autonomy == 0:
            self.singularity_status = "⏳ WEE Phase (Waiting for Engineering Execution)"
        elif self.singularity_value >= 1.0:
            self.singularity_status = "✅✅ Sustained"
        elif self.singularity_value >= self.singularity_threshold:
            self.singularity_status = "✅ Achieved"
        else:
            self.singularity_status = "🔶 Approaching"

    @classmethod
    def from_experiment_data(cls, data: Dict) -> 'TOPO2026_6x5_Metrics':
        """Create metrics from experimental data."""
        runs = data.get('results', [])

        if not runs:
            return cls()

        forgetting_rates = []
        bwt_values = []
        fwt_values = []
        degradations = []
        consistencies = []

        for run in runs:
            forgetting_A = run.get('after_A', 0.0) - run.get('final', {}).get('A', 0.0)
            forgetting_B = run.get('after_B', {}).get('B', 0.0) - run.get('final', {}).get('B', 0.0)
            forgetting_avg = (forgetting_A + forgetting_B) / 2
            forgetting_rates.append(forgetting_avg)

            bwt_A = run.get('final', {}).get('A', 0.0) - run.get('after_A', 0.0)
            bwt_B = run.get('final', {}).get('B', 0.0) - run.get('after_B', {}).get('B', 0.0)
            bwt_avg = (bwt_A + bwt_B) / 2
            bwt_values.append(bwt_avg)

            zero_shot = run.get('zero_shot', {})
            fwt_A = run.get('after_A', 0.0) - zero_shot.get('A', 0.0)
            fwt_B = run.get('after_B', {}).get('B', 0.0) - zero_shot.get('B', 0.0)
            fwt_C = run.get('final', {}).get('C', 0.0) - zero_shot.get('C', 0.0)
            fwt_avg = (fwt_A + fwt_B + fwt_C) / 3
            fwt_values.append(fwt_avg)

            deg_A = run.get('after_A', 0.0) - run.get('final', {}).get('A', 0.0)
            deg_B = run.get('after_B', {}).get('B', 0.0) - run.get('final', {}).get('B', 0.0)
            degradations.append(max(deg_A, deg_B))

            final = run.get('final', {})
            consistency = (final.get('A', 0.0) + final.get('B', 0.0) + final.get('C', 0.0)) / 3
            consistencies.append(consistency)

        # Convert to Python float for JSON serialization
        forgetting_mean = float(np.mean(forgetting_rates) * 100)
        forgetting_std = float(np.std(forgetting_rates) * 100)

        bwt_mean = float(np.mean(bwt_values) * 100)
        bwt_std = float(np.std(bwt_values) * 100)

        fwt_mean = float(np.mean(fwt_values) * 100)
        fwt_std = float(np.std(fwt_values) * 100)

        deg_mean = float(np.mean(degradations) * 100)
        deg_std = float(np.std(degradations) * 100)

        cons_mean = float(np.mean(consistencies) * 100)
        cons_std = float(np.std(consistencies) * 100)

        forgetting_pass = forgetting_mean <= 10.0
        bwt_pass = bwt_mean >= -5.0
        fwt_pass = fwt_mean >= 20.0
        deg_pass = deg_mean <= 5.0
        cons_pass = cons_mean >= 85.0

        five_by_five = all([forgetting_pass, bwt_pass, fwt_pass, deg_pass, cons_pass])

        # Compute Singularity Components
        agi_gate = float(min(1.0, cons_mean / 100 * 0.8))
        memory_fidelity = float(1.0 - (forgetting_mean / 100))
        memory_fidelity_std = float(forgetting_std / 100)
        validation_accuracy = float(1.0 if five_by_five else 0.0)
        forward_transfer = float(1.0 + (fwt_mean / 100))
        forward_transfer_std = float(fwt_std / 100)

        # Compute dI/dt
        improvements = []
        for run in runs:
            zero_shot = run.get('zero_shot', {})
            imp_A = run.get('after_A', 0.0) - zero_shot.get('A', 0.0)
            imp_B = run.get('after_B', {}).get('B', 0.0) - zero_shot.get('B', 0.0)
            imp_C = run.get('final', {}).get('C', 0.0) - zero_shot.get('C', 0.0)
            improvements.extend([imp_A, imp_B, imp_C])

        intelligence_rate = float(np.mean(improvements))

        # Compute efficiency
        model_size = 20.0
        avg_training_time = 3.0
        compute_efficiency = float(model_size / avg_training_time)

        metrics = cls(
            forgetting_rate=forgetting_mean,
            forgetting_std=forgetting_std,
            forgetting_pass=forgetting_pass,
            bwt_corrected=bwt_mean,
            bwt_std=bwt_std,
            bwt_pass=bwt_pass,
            fwt=fwt_mean,
            fwt_std=fwt_std,
            fwt_pass=fwt_pass,
            degradation=deg_mean,
            degradation_std=deg_std,
            degradation_pass=deg_pass,
            consistency=cons_mean,
            consistency_std=cons_std,
            consistency_pass=cons_pass,
            five_by_five_certified=five_by_five,
            agi_gate=agi_gate,
            intelligence_rate=intelligence_rate,
            memory_fidelity=memory_fidelity,
            memory_fidelity_std=memory_fidelity_std,
            validation_accuracy=validation_accuracy,
            forward_transfer=forward_transfer,
            forward_transfer_std=forward_transfer_std,
            compute_efficiency=compute_efficiency,
            raw_metrics={
                'forgetting': {'mean': forgetting_mean, 'std': forgetting_std},
                'bwt': {'mean': bwt_mean, 'std': bwt_std},
                'fwt': {'mean': fwt_mean, 'std': fwt_std},
                'degradation': {'mean': deg_mean, 'std': deg_std},
                'consistency': {'mean': cons_mean, 'std': cons_std}
            },
            run_data=runs
        )

        metrics.compute_singularity()
        return metrics

    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON serialization."""
        def to_serializable(obj):
            if isinstance(obj, bool):
                return bool(obj)
            elif isinstance(obj, (np.integer, np.int64, np.int32)):
                return int(obj)
            elif isinstance(obj, (np.floating, np.float64, np.float32)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: to_serializable(v) for k, v in obj.items()}
            elif isinstance(obj, (list, tuple)):
                return [to_serializable(item) for item in obj]
            elif isinstance(obj, (str, int, float)):
                return obj
            elif obj is None:
                return None
            else:
                return str(obj)

        return {
            'timestamp': self.timestamp,
            'five_by_five': {
                'certified': to_serializable(self.five_by_five_certified),
                'metrics': {
                    'forgetting': {
                        'mean': to_serializable(self.forgetting_rate),
                        'std': to_serializable(self.forgetting_std),
                        'pass': to_serializable(self.forgetting_pass),
                        'threshold': to_serializable(self.forgetting_threshold)
                    },
                    'bwt_corrected': {
                        'mean': to_serializable(self.bwt_corrected),
                        'std': to_serializable(self.bwt_std),
                        'pass': to_serializable(self.bwt_pass),
                        'threshold': to_serializable(self.bwt_threshold)
                    },
                    'fwt': {
                        'mean': to_serializable(self.fwt),
                        'std': to_serializable(self.fwt_std),
                        'pass': to_serializable(self.fwt_pass),
                        'threshold': to_serializable(self.fwt_threshold)
                    },
                    'degradation': {
                        'mean': to_serializable(self.degradation),
                        'std': to_serializable(self.degradation_std),
                        'pass': to_serializable(self.degradation_pass),
                        'threshold': to_serializable(self.degradation_threshold)
                    },
                    'consistency': {
                        'mean': to_serializable(self.consistency),
                        'std': to_serializable(self.consistency_std),
                        'pass': to_serializable(self.consistency_pass),
                        'threshold': to_serializable(self.consistency_threshold)
                    }
                }
            },
            'singularity': {
                'value': to_serializable(self.singularity_value),
                'std': to_serializable(self.singularity_std),
                'status': to_serializable(self.singularity_status),
                'threshold': to_serializable(self.singularity_threshold),
                'components': {
                    'agi_gate': to_serializable(self.agi_gate),
                    'intelligence_rate': to_serializable(self.intelligence_rate),
                    'memory_fidelity': {
                        'mean': to_serializable(self.memory_fidelity),
                        'std': to_serializable(self.memory_fidelity_std)
                    },
                    'validation_accuracy': to_serializable(self.validation_accuracy),
                    'forward_transfer': {
                        'mean': to_serializable(self.forward_transfer),
                        'std': to_serializable(self.forward_transfer_std)
                    },
                    'compute_efficiency': to_serializable(self.compute_efficiency),
                    'autonomy': to_serializable(self.autonomy)
                }
            },
            'raw_metrics': to_serializable(self.raw_metrics),
            'num_runs': to_serializable(len(self.run_data))
        }

    def print_certification(self):
        """Print complete 6×5 certification."""
        print("=" * 80)
        print("TOPO-2026 6×5 CERTIFICATION SYSTEM")
        print("=" * 80)

        print("\n┌─────────────────────────────────────────────────────────────────┐")
        print("│  5×5 CONTINUAL LEARNING METRICS                               │")
        print("├─────────────────────────────────────────────────────────────────┤")
        print(f"│  1. Forgetting        : {self.forgetting_rate:>6.2f}% ± {self.forgetting_std:>5.2f}%  {'✅' if self.forgetting_pass else '❌'}")
        print(f"│  2. BWT (Corrected)   : {self.bwt_corrected:>+6.2f}% ± {self.bwt_std:>5.2f}%  {'✅' if self.bwt_pass else '❌'}")
        print(f"│  3. FWT               : {self.fwt:>+6.2f}% ± {self.fwt_std:>5.2f}%  {'✅' if self.fwt_pass else '❌'}")
        print(f"│  4. Degradation       : {self.degradation:>6.2f}% ± {self.degradation_std:>5.2f}%  {'✅' if self.degradation_pass else '❌'}")
        print(f"│  5. Consistency       : {self.consistency:>6.2f}% ± {self.consistency_std:>5.2f}%  {'✅' if self.consistency_pass else '❌'}")
        print("├─────────────────────────────────────────────────────────────────┤")
        print(f"│  5×5 Status           : {'✅ CERTIFIED' if self.five_by_five_certified else '❌ NOT CERTIFIED'}")
        print("└─────────────────────────────────────────────────────────────────┘")

        print("\n┌─────────────────────────────────────────────────────────────────┐")
        print("│  6TH METRIC: SINGULARITY VALUE                               │")
        print("├─────────────────────────────────────────────────────────────────┤")
        print(f"│  S = {self.singularity_value:.6f} ± {self.singularity_std:.6f}")
        print(f"│  STATUS: {self.singularity_status}")
        print("├─────────────────────────────────────────────────────────────────┤")
        print("│  Components:")
        print(f"│    AGI_gate   : {self.agi_gate:.4f}")
        print(f"│    dI/dt      : {self.intelligence_rate:.4f}")
        print(f"│    M(t)       : {self.memory_fidelity:.4f} ± {self.memory_fidelity_std:.4f}")
        print(f"│    V(t)       : {self.validation_accuracy:.4f}")
        print(f"│    F(t)       : {self.forward_transfer:.4f} ± {self.forward_transfer_std:.4f}")
        print(f"│    C(t)       : {self.compute_efficiency:.4f}")
        print(f"│    Autonomy   : {self.autonomy:.0f} {'✅' if self.autonomy > 0 else '❌'}")
        print("└─────────────────────────────────────────────────────────────────┘")

        print("\n┌─────────────────────────────────────────────────────────────────┐")
        print("│  OVERALL STATUS                                               │")
        print("├─────────────────────────────────────────────────────────────────┤")
        if self.five_by_five_certified and self.singularity_value >= self.singularity_threshold:
            print("│  ✅ COMPLETE 6×5 CERTIFICATION PASSED                      │")
            print("│  ✅ SINGULARITY ACHIEVED                                  │")
        elif self.five_by_five_certified:
            print("│  ✅ 5×5 CERTIFICATION PASSED                               │")
            print("│  ⚠️ SINGULARITY APPROACHING                                │")
        else:
            print("│  ❌ CERTIFICATION FAILED                                  │")
        print("└─────────────────────────────────────────────────────────────────┘")
        print("=" * 80)


# ============================================================================
# 7. MAIN EXECUTION
# ============================================================================

def main():
    """Run the complete 6×5 singularity experiment."""

    print("=" * 80)
    print("TOPO-2026 6×5 SINGULARITY EXPERIMENT")
    print("=" * 80)
    print("\nThis runs the actual GPT-OSS-20B experiment and computes 6×5 metrics.\n")

    # Create configuration
    config = SingularityConfig()

    # Create trainer
    trainer = LLMSingularityTrainer(config)

    # Run experiments
    print("\n[Running Experiments]")
    results = trainer.run_full_experiment()

    # Compute 6×5 metrics from real results
    print("\n" + "=" * 80)
    print("COMPUTING 6×5 METRICS")
    print("=" * 80)

    experiment_data = {'results': results}
    metrics = TOPO2026_6x5_Metrics.from_experiment_data(experiment_data)

    # Print certification
    metrics.print_certification()

    # Save results
    output_dir = "./topo_2026_6x5_results"
    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/6x5_metrics.json", 'w') as f:
        json.dump(metrics.to_dict(), f, indent=2)

    with open(f"{output_dir}/experiment_results.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)

    print(f"\n✅ Results saved to: {output_dir}")
    print(f"   - 6x5_metrics.json")
    print(f"   - experiment_results.json")

    return metrics, results


if __name__ == "__main__":
    metrics, results = main()

TOPO-2026 6×5 SINGULARITY EXPERIMENT

This runs the actual GPT-OSS-20B experiment and computes 6×5 metrics.

[Trainer] Using device: cuda
[Trainer] Loading GPT-OSS-20B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

[Dataset] Loading AG News...
[Dataset] Created 3 tasks
  Task A: 500 train, 200 test
  Task B: 1000 train, 200 test
  Task C: 1000 train, 200 test

[Running Experiments]

RUN 1/5
Learning Rates: embed=5e-03, cls=1e-03

[Zero-shot] Evaluating tasks before training...
  A: 53.50%
  B: 36.50%
  C: 53.50%

[Task A] Training...


[Run 0] Task A: lr_embed=5e-03, lr_cls=1e-03:   0%|          | 0/192 [00:00<?, ?it/s]

...

  Accuracy after A: 88.00%
[Governor] Anchors: [2, 3, 5, 7, 11, 13], Safety: 0.9785

[Task B] Training...


[Run 4] Task B: lr_embed=2e-03, lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  Accuracy A after B: 88.50%
  Accuracy B after B: 79.50%

[Task C] Training...


[Run 4] Task C: lr_embed=2e-03, lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  Final A: 87.50%
  Final B: 79.50%
  Final C: 90.50%

──────────────────────────────────────────────────
RUN 4 COMPLETE (195.7s)
  Memory Fidelity: 0.9975
  Forward Transfer: 1.3817
  Consistency: 0.8583
──────────────────────────────────────────────────

EXPERIMENT COMPLETE - Total time: 16.3 minutes

COMPUTING 6×5 METRICS
TOPO-2026 6×5 CERTIFICATION SYSTEM

┌─────────────────────────────────────────────────────────────────┐
│  5×5 CONTINUAL LEARNING METRICS                               │
├─────────────────────────────────────────────────────────────────┤
│  1. Forgetting        :   1.30% ±  2.93%  ✅
│  2. BWT (Corrected)   :  -1.30% ±  2.93%  ✅
│  3. FWT               : +37.77% ±  3.11%  ✅
│  4. Degradation       :   1.90% ±  3.20%  ✅
│  5. Consistency       :  85.20% ±  2.42%  ✅
├─────────────────────────────────────────────────────────────────┤
│  5×5 Status           : ✅ CERTIFIED
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────